# Bounded Self-Instruct → Qwen3-8B SFT → Lighteval loop

GPT-6 Luna is the parent model, accessed through the OpenAI Responses API. GPU workers load quantized student replicas with the current LoRA adapter for each phase. The notebook coordinates API calls and holds no model weights. Smoke mode generates teacher-checked synthetic pairs and evaluates with native Lighteval metrics. Full mode uses a mixed teacher/template curriculum, reviewed validation data for selection, and a separate final test with regression and promotion gates.

Student predictions are diagnostics. The student's forward pass in SFTTrainer is trained against parent completion tokens; incorrect sampled student responses are never used as labels. This notebook does not claim independent medical validation. Run cells in order on a CUDA GPU with Internet enabled.

Kaggle T4 x2 mode: restart the kernel, expose GPUs 0 and 1, and keep the notebook free of model weights. Separate workers use DDP for training and batched parallel inference. Existing adapter checkpoints and completed data remain reusable; floating-point results may differ across hardware layouts.


In [ ]:
from pathlib import Path
import os
import subprocess

# Configure CUDA before any torch import. Restart first if CUDA was already used.
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"

# Fresh Kaggle sessions need the complete repository before project discovery.
# Outside Kaggle, open this notebook from your existing complete checkout.
KAGGLE_REPO = Path("/kaggle/working/Health-Insurance-SLM")
if Path("/kaggle/working").is_dir():
    if not (Path.cwd() / "requirements-training.txt").is_file() and not KAGGLE_REPO.exists():
        subprocess.run([
            "git", "clone", "--branch", "Distillation",
            "https://github.com/j6antony/Health-Insurance-SLM.git", str(KAGGLE_REPO),
        ], check=True)
    elif KAGGLE_REPO.exists():
        print("Reusing existing checkout:", KAGGLE_REPO)
    print("Cloning does not include unpushed local changes. Existing checkouts are not automatically updated.")


In [ ]:
from pathlib import Path
import os
import sys

# Two independent GPU workers: set BEFORE importing torch.
# Restart the kernel first if CUDA was initialized in an earlier cell.
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"

# Attach/clone the complete repository, not just this notebook.
# Override PROJECT_ROOT explicitly if your checkout is elsewhere.
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "requirements-training.txt").is_file():
    candidates = []
    for parent in (Path("/kaggle/working"), Path("/kaggle/input")):
        if parent.exists():
            candidates.extend(parent.rglob("requirements-training.txt"))
    candidates = [p.parent for p in candidates if (p.parent / "run_self_instruct.py").is_file()]
    if len(candidates) != 1:
        raise RuntimeError("Set PROJECT_ROOT to the complete repository checkout, then rerun this cell.")
    PROJECT_ROOT = candidates[0]
PROJECT_ROOT = PROJECT_ROOT.resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
REQUIREMENTS = str(PROJECT_ROOT / "requirements-training.txt")
print("Project:", PROJECT_ROOT)

# Catch an old or incomplete clone before imports, downloads, or paid calls.
required_files = [
    "distillation_loop/preflight.py", "distillation_loop/review.py",
    "distillation_loop/quality.py", "distillation_loop/trusted_templates.py",
    "self_instruct_pipeline/rule_checks.py",
]
missing_files = [name for name in required_files if not (PROJECT_ROOT / name).is_file()]
if missing_files:
    raise RuntimeError(
        "Repository is outdated or incomplete: " + ", ".join(missing_files)
        + ". Upload/sync the complete updated repository, restart the kernel, and rerun from the top."
    )


In [ ]:
%pip install -q -r "{REQUIREMENTS}"
# If an earlier cell imported incompatible library versions, restart the kernel
# once after installation, then rerun from the top.


After dependency installation, restart the kernel if required and rerun setup cells from the top. Do not insert an automatic kernel-exit cell into Run All. Enable Kaggle T4 ×2 and Internet before continuing.


In [ ]:
import os

# Only the OpenAI parent needs a credential. Qwen3-8B loads anonymously below.
if not os.environ.get("OPENAI_API_KEY"):
    try:
        from kaggle_secrets import UserSecretsClient
        os.environ["OPENAI_API_KEY"] = UserSecretsClient().get_secret("OPENAI_API_KEY").strip()
    except Exception as exc:
        raise RuntimeError("Add OPENAI_API_KEY to Kaggle Secrets and enable notebook access.") from None
if not os.environ.get("OPENAI_API_KEY", "").strip():
    raise RuntimeError("OPENAI_API_KEY is empty.")
print("OpenAI credential loaded. Hugging Face login is not required.")

# Optional durable cost history. First run supabase/api_usage.sql in Supabase.
# Add BOTH values in Kaggle Secrets and enable notebook access to each.
for name in ("SUPABASE_URL", "SUPABASE_SECRET_KEY"):
    if not os.environ.get(name):
        try:
            from kaggle_secrets import UserSecretsClient
            os.environ[name] = UserSecretsClient().get_secret(name).strip()
        except Exception:
            pass
from self_instruct_pipeline.usage_cloud import SupabaseLedger
cloud_ledger = SupabaseLedger.from_environment()
if cloud_ledger:
    cloud_ledger.check()
    print("Supabase cost ledger connected. Calls will sync automatically.")
else:
    print("Cost logging is LOCAL ONLY. Add Supabase secrets for history across sessions.")

# Automatic private adapter datasets after each training round.
# Add KAGGLE_USERNAME (your account name) and either KAGGLE_API_TOKEN or KAGGLE_KEY.
for name in ("KAGGLE_USERNAME", "KAGGLE_API_TOKEN", "KAGGLE_KEY"):
    if not os.environ.get(name):
        try:
            from kaggle_secrets import UserSecretsClient
            os.environ[name] = UserSecretsClient().get_secret(name).strip()
        except Exception:
            pass
BACKUP_ADAPTERS_TO_KAGGLE = True
if BACKUP_ADAPTERS_TO_KAGGLE:
    owner = os.environ.get("KAGGLE_USERNAME", "").strip()
    if not owner:
        raise RuntimeError("Add your account name as KAGGLE_USERNAME in Kaggle Secrets for adapter backups.")
    os.environ["KAGGLE_ADAPTER_DATASET"] = owner + "/health-insurance-slm-adapters"
    from distillation_loop.kaggle_backup import check_backup_setup
    check_backup_setup()
    print("Private Kaggle adapter backup enabled.")
else:
    os.environ.pop("KAGGLE_ADAPTER_DATASET", None)


In [ ]:
# Restart the kernel before switching from the old single-GPU notebook.
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"
import torch
from transformers import AutoTokenizer
from distillation_loop.evaluation import require_training_versions
require_training_versions()
assert torch.cuda.is_available() and torch.cuda.device_count() == 2, "Enable Kaggle T4 x2 and restart the kernel."
MODEL_NAME = "Qwen/Qwen3-8B"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=False)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
student = None  # Worker processes own model weights; keep notebook GPU memory free.
print("Visible GPUs:", torch.cuda.device_count())
print("GPU workers:", [torch.cuda.get_device_name(i) for i in range(2)])


## Loop settings

Leave **TEST_RUN = True** for the first GPU smoke run: one round, eight training pairs, one synthetic development example per task, and at most two optimizer steps. This checks integration and cannot promote a release adapter. The existing API budget caps remain active; exhausting a cap stops the run safely and does not establish model quality.

A full run uses reviewed validation and final-test files with at least 30 examples per task in each split (540 + 540). Reasoning references need reviewed rubrics; unfamiliar student wording pauses for independent adjudication. Final promotion requires quality thresholds, improvement and regression checks. See `data/evaluation/README.md` for the file and review contracts.

Use the complete updated repository, not just this notebook. A GitHub clone only contains changes already pushed to that branch. The versioned output folders below separate this implementation from older scoring contracts. Restart the kernel before switching runs or updating Python modules.


In [ ]:
from self_instruct_pipeline.config import load_config
from self_instruct_pipeline.persistence import atomic_json
from distillation_loop.core import load_loop_config
from distillation_loop.dual_gpu import DualGPURuntime

# Keep True for your first integration test. Change only for a NEW full-run folder.
TEST_RUN = True

parent_config = load_config(PROJECT_ROOT / "config/self_instruct.json")
loop_config = load_loop_config(str(PROJECT_ROOT / "config/distillation_loop.json"))

# Editable before starting a new run:
loop_config["max_rounds"] = 5
loop_config["thresholds"] = {
    "macro_task_score": 0.90,
    "minimum_task_score": 0.80,
    "schema_evidence_valid_rate": 0.98,
}
RUN_DIR = (Path("/kaggle/working/distillation_loop_reviewed_v2") if Path("/kaggle/working").exists()
           else PROJECT_ROOT / "outputs/distillation_loop_reviewed_v2")
if TEST_RUN:
    from distillation_loop.smoke import smoke_configs
    RUN_DIR = RUN_DIR.with_name("distillation_smoke_v2")
    parent_config, loop_config = smoke_configs(parent_config, loop_config, RUN_DIR)
print("SMOKE TEST" if TEST_RUN else "FULL RUN", "Output:", RUN_DIR)
RUN_DIR.mkdir(parents=True, exist_ok=True)
if cloud_ledger:
    from self_instruct_pipeline.usage_cloud import sync_usage_history
    print(sync_usage_history(RUN_DIR))  # Import prior local costs before new paid calls.
loop_config["output_dir"] = str(RUN_DIR)
# Validate notebook overrides using the same configuration loader.
atomic_json(RUN_DIR / "requested_loop_config.json", loop_config)
loop_config = load_loop_config(str(RUN_DIR / "requested_loop_config.json"))
# Read-only data/configuration check before model workers or paid calls.
from distillation_loop.preflight import readiness
readiness_report = readiness(parent_config, loop_config, check_runtime=False)
print("Readiness:", readiness_report)
if not readiness_report["ready"]:
    raise RuntimeError("Resolve readiness issues above before running the experiment.")
student_runtime = DualGPURuntime(tokenizer, loop_config, MODEL_NAME, batch_size=2)
student_runtime.prepare()  # Checks two GPUs; workers load/train their own LoRA models.
print("Max rounds:", loop_config["max_rounds"])
print("Thresholds:", loop_config["thresholds"])


In [ ]:
# Optional request preview: this cell makes no parent API calls.
import copy
from self_instruct_pipeline.pipeline import Pipeline
preview_config = copy.deepcopy(parent_config)
preview_config["paths"]["output_dir"] = str(RUN_DIR / "preview")
preview = Pipeline(preview_config).plan()
print("Task quotas:", preview["task_quotas"])
print("Seed demonstrations:", preview["seed_ids"])

# Free local GPU check before the paid run. It checks generation, not accuracy.
probes = student_runtime.infer_many([
    {"instruction": "Reply with the JSON object {\"ok\":true}.", "input": {}},
    {"instruction": "Reply with the JSON object {\"ok\":true}.", "input": {}},
])
if any(probe.get("error") or not probe.get("generated_tokens") for probe in probes):
    raise RuntimeError("Student GPU preflight failed; inspect the worker output.")
print("Both GPU workers generated responses. No parent API calls made.")


## Run or resume

This cell spends parent API credits and performs GPU training. Rerun it with the same configuration/output directory to resume. Completed training and frozen evaluation predictions are reused. A pending independent review raises `ReviewRequired`; fill that evaluation directory's `adjudications.jsonl` from its `review_queue.jsonl`, then rerun the run cell. Do not reset budgets or delete state to bypass a stop.

Smoke runs restore the best smoke checkpoint but cannot promote it. Reviewed full runs restore `release_checkpoint`: the selected adapter only if final quality/improvement/regression gates pass, otherwise the initial adapter. Inspect `promotion` and `final_comparison`, not only validation scores.

Restart the kernel before switching to `TEST_RUN = False`. Full runs require reviewed benchmark files; the readiness check below the settings catches missing data before workers load models or paid generation starts.


In [ ]:
from distillation_loop.controller import run_loop
from self_instruct_pipeline.budget import APIBudget

# Shared across development, training generation, audits, and retries.
# Counts persist across kernel restarts; rerunning does not reset the budget.
budget = APIBudget(RUN_DIR / "api_budget",
                   max_calls=120 if TEST_RUN else 2000,
                   max_cost_usd="0.50" if TEST_RUN else "5.00")
with budget.activate():
    result = run_loop(student_runtime, parent_config, loop_config)
student = student_runtime.model
print("Stop reason:", result["status"])
print("Completed rounds:", result["completed_rounds"])
print("Best validation checkpoint:", result["best_checkpoint"])
print("Promotion:", result.get("promotion"))
print("Release checkpoint:", result.get("release_checkpoint", "No release: synthetic smoke only"))
print("Best validation macro score:", result["best_report"]["macro_task_score"])
print("Budget accounting:", budget.state)


In [ ]:
# Inspect history without rerunning inference or training.
for entry in result["history"]:
    scores = entry["scores"]
    print({"round": entry["round"], "macro": scores["macro_task_score"],
           "minimum_task": scores["minimum_task_score"],
           "schema_evidence_valid": scores["schema_evidence_valid_rate"]})


In [ ]:
# Refresh the combined usage report, including calls from earlier sessions.
from self_instruct_pipeline.usage import export_usage
usage_summary = export_usage(RUN_DIR)
print(usage_summary)


In [ ]:
# Optional: export ALL saved Supabase history, even from previous Kaggle sessions.
from self_instruct_pipeline.usage_cloud import export_cloud_usage
if cloud_ledger:
    print(export_cloud_usage(RUN_DIR / "api_usage_all_runs.csv"))
# To export only this run:
# run_id = json.loads((RUN_DIR / "api_usage_run.json").read_text())["run_id"]
# print(export_cloud_usage(RUN_DIR / "api_usage_this_run.csv", run_id=run_id))
